# Objects

In Python, everything is an object. 

In [1]:
a = 1
def hi(): ...
class Dog: ...

print(isinstance(a, object))
print(isinstance(hi, object))
print(isinstance(Dog, object))

True
True
True


Even functions and classes are objects.

**But what is an object?**

Conceptually, an object is something with an identity, a type, and a value

- Identity makes the object distinct from every other object
- Type defines the object's behaviour
- Value holds the object's data

CPython implements this as a C struct. Every object’s struct starts with the same header consisting of

- A reference count (for memory management)
- A pointer to its type

it then adds its own fields to store its value. Note that identity isn't explicitly stored anywhere - CPython uses the struct's memory address as the identity.

For example,

- `5` is an object that contains a reference count, a pointer to the `int` type, and its value, which is the number 5 in binary plus its sign and length.
- `int` is an object that contains a reference count, a pointer to the `type` type, and its value, which is a bunch of stuff that defines how int instances behave.

In [2]:
x = 5

# identity, type, value
print(id(x), type(x), x)

# the type chain: 5 → int → type → type
print(type(5), type(int), type(type))

198953870842176 <class 'int'> 5
<class 'int'> <class 'type'> <class 'type'>


# Namespaces

In Python, every "variable" is a name, which is bound to an object within a namespace.

Conceptually, a namespace is a mapping from names to objects. CPython implements them as dict attributes of objects (except for function locals), for example:

- The built-in namespace is stored in `builtins.__dict__`
- The module namespace is stored in `sys.modules[__name__].__dict__` or `globals()`
- The class namespace is stored in `Class.__dict__`
- The instance namespace is stored in `obj.__dict__`
- The function local level in internal slots for better performance

In [3]:
import builtins

x = "global x"

class Parent:
    p = "from Parent"

class Child(Parent):
    c = "from Child"
    def __init__(self):
        self.i = "from instance"

obj = Child()
obj.hi = "hi"

print("'len' in builtins.__dict__: ", "len" in builtins.__dict__)
print("'x'   in globals():         ", "x" in globals())
print("'p'   in Parent.__dict__:   ", "p" in Parent.__dict__)
print("'p'   in Child.__dict__:    ", "p" in Child.__dict__)
print("'c'   in Child.__dict__:    ", "c" in Child.__dict__)
print("'c'   in obj.__dict__:      ", "c" in obj.__dict__)
print("obj.__dict__:               ", obj.__dict__)

'len' in builtins.__dict__:  True
'x'   in globals():          True
'p'   in Parent.__dict__:    True
'p'   in Child.__dict__:     False
'c'   in Child.__dict__:     True
'c'   in obj.__dict__:       False
obj.__dict__:                {'i': 'from instance', 'hi': 'hi'}


When you use a name, CPython follows a fixed lookup order

For bare names (`x`), it uses the LEGB rule

- Local: the local namespace of the current function calls
- Enclosing: the local namespace of any outer function calls
- Global: the module namespace
- Built-in: the built-in namespace

For dotted names (`obj.x`), it searches through

- Instance namespace
- Class namespace
- Parent classes' namespaces

**But what does 'bound' actually mean?**

Let's build a mental picture compared to other types of variables:

- In C, a plain variable holds a value
- In C, a pointer holds a memory address
- In C++, a reference is like a C pointer that automatically dereferences, but it can never be re-pointed
- In Python, a name is like a C++ reference, but it can be re-pointed via the assignment operator.